# 08 — GNN training

T10.1 loads the raw graph cache and creates the reproducible train/validation scaffold splits supplied by TDC. Normalizer fitting, DataLoaders, and training are intentionally deferred to later tasks.

In [ ]:
from pathlib import Path

from tdc.benchmark_group import admet_group

from src.gnn.config import GNNConfig
from src.gnn.dataset import load_graph_dataset, select_graph_subset
from src.gnn.normalizer import GraphFeatureNormalizer


In [ ]:
REPO_ROOT = Path.cwd().resolve().parent
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
config = GNNConfig()
GRAPH_PATH = (
    REPO_ROOT
    / "data"
    / "graphs"
    / "aqsoldb"
    / config.graph_config_id
    / "graphs.pt"
)
print("graph cache:", GRAPH_PATH)
print("graph_config_id:", config.graph_config_id)


In [ ]:
if not GRAPH_PATH.is_file():
    raise FileNotFoundError(
        f"Raw graph cache not found at {GRAPH_PATH}. Run notebook 07 first."
    )

graph_artifact = load_graph_dataset(str(GRAPH_PATH), config=config)
graphs = graph_artifact.graphs
if not graphs:
    raise ValueError("Raw graph cache contains no graphs.")

graph_by_id = {}
for graph in graphs:
    graph_id = str(graph.mol_id)
    if graph_id in graph_by_id:
        raise ValueError(f"Duplicate graph mol_id: {graph_id}")
    graph_by_id[graph_id] = graph
print("loaded raw graphs:", len(graphs))


In [ ]:
group = admet_group(path=str(RAW_DATA_DIR))
benchmark_name = "Solubility_AqSolDB"
benchmark = group.get(benchmark_name)
test_records = benchmark["test"]
test_ids = {str(value) for value in test_records["Drug_ID"]}

def split_ids(frame):
    if "Drug_ID" in frame.columns:
        return [str(value) for value in frame["Drug_ID"]]
    smiles_to_id = {str(graph.smiles): graph_id for graph_id, graph in graph_by_id.items()}
    missing_smiles = [str(value) for value in frame["Drug"] if str(value) not in smiles_to_id]
    if missing_smiles:
        raise ValueError(f"scaffold split contains unknown SMILES: {missing_smiles[:3]}")
    return [smiles_to_id[str(value)] for value in frame["Drug"]]


In [ ]:
SEEDS = (0, 1, 2, 3, 4)
scaffold_splits = {}

for seed in SEEDS:
    train_frame, validation_frame = group.get_train_valid_split(
        benchmark=benchmark_name,
        split_type="default",
        seed=seed,
    )
    train_ids = split_ids(train_frame)
    validation_ids = split_ids(validation_frame)
    train_id_set = set(train_ids)
    validation_id_set = set(validation_ids)
    if train_id_set & validation_id_set:
        raise ValueError(f"seed {seed} has overlapping train/validation IDs.")
    if (train_id_set | validation_id_set) & test_ids:
        raise ValueError(f"seed {seed} leaks test IDs into train/validation.")
    if not (train_id_set | validation_id_set) <= set(graph_by_id):
        raise ValueError(f"seed {seed} references IDs absent from graph cache.")

    scaffold_splits[seed] = {
        "train_ids": train_ids,
        "validation_ids": validation_ids,
        "train_graphs": select_graph_subset(graphs, train_ids),
        "validation_graphs": select_graph_subset(graphs, validation_ids),
    }

print("scaffold seeds loaded:", sorted(scaffold_splits))
for seed, split in scaffold_splits.items():
    print(seed, len(split["train_graphs"]), len(split["validation_graphs"]))


In [ ]:
normalizers = {}
for seed, split in scaffold_splits.items():
    normalizers[seed] = (
        GraphFeatureNormalizer.from_config(config)
        .fit(split["train_graphs"])
    )
    if not normalizers[seed].fitted:
        raise RuntimeError(f"normalizer for seed {seed} is not fitted.")

print("train-fitted normalizers:", sorted(normalizers))
